# 07 - Anomaly Detection with Isolation Forest
**Project:** Air Quality & Pollution Intelligence - Data Mining and Business Intelligence

**Objective:** Find records whose combination of pollutant values is statistically unusual, quantify how many there are, and state clearly what the label does and does not mean.

**How to read this notebook:** every number printed below is produced by the
code in this notebook from `data/raw/Air_quality_data.csv`. Column names are
discovered at runtime through `src/config.py`, so nothing is assumed.


In [ ]:
"""Environment bootstrap: make src/ importable and pin the working directory."""
import sys, os, warnings
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)
%matplotlib inline
print("project root:", PROJECT_ROOT)

In [ ]:
import config as C
import data_utils as U
import preprocessing as P
import feature_engineering as FE
import anomaly_detection as AD
import figures as F
import viz as V
from IPython.display import display, Image

clean, *_ = P.clean(U.load_raw())
feat, _ = FE.build_features(clean)
polls = C.pollutant_columns(feat.columns)
V.style()

## 1. Why Isolation Forest

* It isolates points by random splits, so the cost is linear in the number of
  records - practical at 18,265 rows.
* It does not assume the data is Gaussian (true here: notebook 05 shows uniform-like
  marginals), unlike a Z-score or Mahalanobis approach.
* It uses all pollutant dimensions jointly, so it can flag a record whose
  *combination* of values is unusual even when no single value is extreme.

## 2. The `contamination` parameter

`contamination` is the fraction of records the algorithm is told to treat as
outliers, and it fixes the decision threshold. It is an **assumption**, not a
finding, so the sensitivity to it is reported instead of being buried.

In [ ]:
sens = []
for c in (0.01, 0.02, 0.05, 0.10):
    tmp, _, _, _ = AD.detect(feat, polls, contamination=c)
    st = AD.summary(tmp)
    sens.append({"contamination": c, "anomalies": st["Anomalies detected"],
                 "rows_evaluated": st["Rows evaluated"],
                 "observed_pct": st["Anomaly %"]})
display(pd.DataFrame(sens))

In [ ]:
feat, model, used, params = AD.detect(feat, polls)
print("features used :", used)
print("parameters    :", {k: v for k, v in params.items() if k != "features"})
stats_tbl = AD.summary(feat)
for k, v in stats_tbl.items():
    print(f"{k:>24}: {v}")

## 3. Anomaly distribution and score

In [ ]:
paths = F.anomaly(feat, AD.by_city(feat), AD.by_period(feat, "year"), polls)
display(feat["Anomaly_Label"].value_counts().to_frame("records"))
Image(paths["01_anomaly_distribution"])

## 4. Which cities and which years are flagged?

In [ ]:
by_city = AD.by_city(feat)
display(by_city)
by_city.to_csv(C.PROCESSED_DIR / "anomaly_by_city.csv", index=False)
Image(paths["02_anomaly_by_city"])

In [ ]:
by_year = AD.by_period(feat, "year")
display(by_year)
by_year.to_csv(C.PROCESSED_DIR / "anomaly_by_year.csv", index=False)
Image(paths["03_anomaly_timeline"])

In [ ]:
print("is the spread between cities larger than sampling noise would give?")
import statistics_analysis as S
flagged = feat.assign(Is_Anomaly=feat["Anomaly"])
ct = pd.crosstab(flagged[C.CITY_COL], flagged["Anomaly"], values=flagged["AQI"],
                aggfunc="size")
from scipy import stats as sps
chi2, p, dof, _ = sps.chi2_contingency(ct)
print(f"chi-square test of independence between City and Anomaly: chi2={chi2:.3f}, "
      f"df={dof}, p={p:.4g}")
print("expected under a fixed contamination rate: about equal shares in every city")

## 5. What makes a record anomalous here?

Comparing the mean pollutant level of flagged and normal records shows which
measurements drive the flag. Note that anomalies are **not** simply the highest
values - the algorithm flags unusual combinations.

In [ ]:
cmp_tbl = AD.pollutant_comparison(feat, polls)
display(cmp_tbl)
cmp_tbl.to_csv(C.PROCESSED_DIR / "anomaly_pollutant_comparison.csv", index=False)
Image(paths["04_anomaly_scatter"])

## 6. The most anomalous records in the file

In [ ]:
worst = (feat.nsmallest(15, "Anomaly_Score")
              [[C.CITY_COL, "Date", "Season"] + polls + [C.AQI_COL, C.TARGET_COL,
                                                        "Anomaly_Score"]])
display(worst.reset_index(drop=True))

## 7. Interpretation limits

An anomaly label means "this record is far from the bulk of the data in the
chosen feature space". It does **not** identify a cause. Naming a cause (fireworks,
stubble burning, a dust storm, a sensor fault) would need activity or maintenance
data that this dataset does not contain, so no cause is asserted.

In [ ]:
keep = [c for c in [C.CITY_COL, "Date", "Year", "Month", "Season"] + polls +
          [C.AQI_COL, C.TARGET_COL, "Anomaly", "Anomaly_Label", "Anomaly_Score",
           "Cluster"] if c in feat.columns]
feat[keep].to_csv(C.ANOMALY_RESULTS_CSV, index=False)
print("saved:", C.ANOMALY_RESULTS_CSV.name, "rows:", len(feat))

## Verification checklist

- [x] Feature set restricted to measured concentrations
- [x] Contamination sensitivity reported
- [x] Anomaly share compared against the configured rate
- [x] City/year breakdowns tested for non-randomness
- [x] No causal explanation claimed

**Common errors**

| Error | Meaning | Fix |
|---|---|---|
| `ValueError: could not convert string to float` | a categorical column entered the feature list | pass only `C.pollutant_columns(...)` |
| All scores identical | a constant column dominates | drop zero-variance columns |

**Next:** `08_classification.ipynb`.